In [1]:
import numpy as np
import matplotlib.pyplot as plt
import utils
import scipy
import plotly.colors as pc
import ot
import pandas as pd
import plotly.graph_objects as go
%load_ext autoreload
%autoreload 2

In [2]:
# This function is in utils but I slightly modified it here so that the poster graphics are a bit cleaner (no axes and gridlines for example)

def plot_arr(arr, color=None, colorbar=False, size=None, cam = dict(x=2.5, y=2.5, z=2.5)):
    fig = go.Figure()

    x = arr[:, 0]
    y = arr[:, 1]
    z = arr[:, 2]

    if color is None:
        color = z

    if size is None:
        size = 2

    fig.add_trace(
        go.Scatter3d(
            x=x,
            y=y,
            z=z,
            mode="markers",
            marker=dict(
                color=color,
                size=size,
                showscale=colorbar,
                colorscale="Plotly3",
                opacity=1.0,
            ),
        )
    )

    fig.update_layout(
        scene=dict(
            aspectmode="data",
            xaxis=dict(
                visible=False,
                showbackground=False,
                showgrid=False,
                zeroline=False,
                showticklabels=False,
                title=""
            ),
            yaxis=dict(
                visible=False,
                showbackground=False,
                showgrid=False,
                zeroline=False,
                showticklabels=False,
                title=""
            ),
            zaxis=dict(
                visible=False,
                showbackground=False,
                showgrid=False,
                zeroline=False,
                showticklabels=False,
                title=""
            ),
            camera=dict(eye=cam),
        ),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=0, r=0, t=0, b=0),
        width=600,
        height=1000,
        showlegend=False,
    )

    return fig

def values_to_dark_colors(values, colorscale="Inferno", dark_range=(0.0, 0.6)):
    import numpy as np
    import plotly.colors as pc

    values = np.asarray(values, dtype=float)
    vmin = np.nanmin(values)
    vmax = np.nanmax(values)

    if vmax == vmin:
        t = np.zeros_like(values)
    else:
        t = (values - vmin) / (vmax - vmin)

    lo, hi = dark_range
    t = lo + t * (hi - lo)

    return pc.sample_colorscale(colorscale, t)



In [3]:
p = "datasets/action_smplx_models/male_run.npz"
points, faces = utils.sampled_verts_from_path(p, idx = 60, n_points = 1000, return_faces=True)
regions = utils.faces_to_regions(faces)
color = ["red" if reg != "left_shin" else "green" for reg in regions]
plot_arr(points, cam = dict(x=0, y=2.5, z=1), color = values_to_dark_colors(points[:,2], colorscale="Magma"), size = 5)

In [4]:
p = "datasets/action_smplx_models/male2_Calibration_stageii.npz"
points, faces = utils.sampled_verts_from_path(p, n_points = 10000, return_faces = True)
regions, colors = utils.faces_to_regions(faces, return_colors = True)
colors = ["#F22424" if c == "#B6E880" else "#006E0D" if c == "#00CC96" else "#006D82" if c == "#19D3F3" else c for c in colors]
plot_arr(points, color = colors, cam = dict(x=0, y=3.2, z=1))


In [5]:
p = "datasets/action_smplx_models/female_run.npz"
points, faces = utils.sampled_verts_from_path(p, n_points = 1000, return_faces = True, idx = 10)
regions, colors = utils.faces_to_regions(faces, return_colors = True)
colors = ["#F22424" if c == "#B6E880" else "#006E0D" if c == "#00CC96" else "#006D82" if c == "#19D3F3" else c for c in colors]
plot_arr(points, color = colors, cam = dict(x=0, y=2, z=1), size = 5)


In [6]:
np.unique(colors)

array(['#006D82', '#006E0D', '#1f77b4', '#2ca02c', '#636EFA', '#8c564b',
       '#9467bd', '#AB63FA', '#EF553B', '#F22424', '#FECB52', '#FF6692',
       '#FF97FF', '#FFA15A', '#d62728', '#ff7f0e'], dtype='<U7')

In [7]:
colors = ["#F22424" if c == "#B6E880" else "#006E0D" if c == "#00CC96" else "#006D82" if c == "#19D3F3" else c for c in colors]

In [8]:
plot_arr(points, color = colors, cam = dict(x=0, y=2, z=1))

In [9]:
def plot_3d_points_and_connections_region_matched(points1, points2, faces1, faces2, G, switch_yz = False, plot_both = True, width = 600, height = 1000,  cam = dict(x=2.5, y=2.5, z=2.5)):
    """
    Given points1, points2, and G, plot the points and lines between matching points. If switch_xz is true then this will switch the x and z coordinates before plotting (since by default in the mocap data the x is the vertical axis).
    points1, points2: Nx3 arrays
    G: NxN array
    switch_xz: Boolean
    """
    if points1.shape[0] != points2.shape[0]:
        raise ValueError("Point clouds are not the same length")

    if G.shape[0] != G.shape[1]:
        raise ValueError("Matching matrix is not square")

    if G.shape[0] != points1.shape[0]:
        raise ValueError("Matching matrix dimensions don't match point cloud dimensions")

    if np.count_nonzero(G) > points1.shape[0]:
        raise ValueError("Matching has too many nonzero entries")

    if np.count_nonzero(G) < points1.shape[0]:
        raise ValueError("Matching has too few nonzero entries")

    print("Region accuracy adjusted:", utils.region_accuracy_adjusted(G, faces1, faces2))

    x_ind = 0
    if switch_yz:
        y_ind = 2
        z_ind = 1
    else:
        y_ind = 1
        z_ind = 2

    # Ensure numpy arrays
    points1 = np.asarray(points1)
    points2 = np.asarray(points2)
    G = np.asarray(G)

    fig = go.Figure()

    regions1 = utils.faces_to_regions(faces1)
    regions2 = utils.faces_to_regions(((G / G.max()) @ faces2).astype(int))
    color1 = ["red" if regions1[i] != regions2[i] else "green" for i in range(len(regions1))]

    # Plot first set of 3D points
    fig.add_trace(go.Scatter3d(
        x=points1[:, x_ind], y=points1[:, y_ind], z=points1[:, z_ind],
        mode='markers',
        marker=dict(size=5, color=color1),
        name='Points 1'
    ))

    if plot_both:
        regions1 = utils.faces_to_regions(((G.T / G.max()) @ faces1).astype(int))
        regions2 = utils.faces_to_regions(faces2)
        color2 = ["red" if regions1[i] != regions2[i] else "green" for i in range(len(regions1))]

    
        # Plot second set of 3D points
        fig.add_trace(go.Scatter3d(
            x=points2[:, x_ind], y=points2[:, y_ind], z=points2[:, z_ind],
            mode='markers',
            marker=dict(size=5, color= color2),
            name='Points 2'
        ))


        # Draw connections for nonzero G[i, j]
        for i in range(G.shape[0]):
            for j in range(G.shape[1]):
                if G[i, j] != 0:
                    p1 = points1[i]
                    p2 = points2[j]
                    fig.add_trace(go.Scatter3d(
                        x=[p1[x_ind], p2[x_ind]],
                        y=[p1[y_ind], p2[y_ind]],
                        z=[p1[z_ind], p2[z_ind]],
                        mode='lines',
                        line=dict(color="gray", width=2),
                        showlegend=False,
                        opacity=0.1
                    ))

    # Layout styling
    fig.update_layout(
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data'
        ),
        title='3D Points with Connections',
        template='plotly_white',
        width = width,
        height = height
    )

    fig.update_layout(
        scene=dict(
            aspectmode="data",
            xaxis=dict(
                visible=False,
                showbackground=False,
                showgrid=False,
                zeroline=False,
                showticklabels=False,
                title=""
            ),
            yaxis=dict(
                visible=False,
                showbackground=False,
                showgrid=False,
                zeroline=False,
                showticklabels=False,
                title=""
            ),
            zaxis=dict(
                visible=False,
                showbackground=False,
                showgrid=False,
                zeroline=False,
                showticklabels=False,
                title=""
            ),
            camera=dict(eye=cam),
        ),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=0, r=0, t=0, b=0),
        width=600,
        height=1000,
        showlegend=False,
    )
    return fig



In [10]:
points1, faces1 = utils.sampled_verts_from_path("datasets/action_smplx_models/male_run.npz", idx = 0, return_faces=True)
points2, faces2 = utils.sampled_verts_from_path("datasets/action_smplx_models/male_run.npz", idx = 60, return_faces=True)
a = np.ones(1000) / 1000
b = np.ones(1000) / 1000
M_baseline = ot.dist(points1, points2)
G_baseline = ot.solve(M_baseline, a, b).plan
aug1 = utils.left_right_augmentation(points1, points2.mean(axis = 0) - points1.mean(axis = 0))
aug2 = utils.left_right_augmentation(points2, points2.mean(axis = 0) - points1.mean(axis = 0))
M_aug = ot.dist(aug1, aug2)
G_aug = ot.solve(M_aug, a, b).plan

In [11]:
plot_3d_points_and_connections_region_matched(points1, points2, faces1, faces2, G_baseline, cam = dict(x = -2, y = 0.5, z = 1))

Region accuracy adjusted: 0.5093360995850622


In [12]:
plot_3d_points_and_connections_region_matched(points1, points2, faces1, faces2, G_aug, cam = dict(x = -2, y = 0.5, z = 1))

Region accuracy adjusted: 0.6431535269709544


In [13]:
def plot_specific_region_connections(points1, points2, faces1, faces2, G, region_label, width = 600, height = 1000, cam = dict(x = 2, y = 2, z = 1), size = 2):
    if points1.shape[0] != points2.shape[0]:
        raise ValueError("Point clouds are not the same length")

    if G.shape[0] != G.shape[1]:
        raise ValueError("Matching matrix is not square")

    if G.shape[0] != points1.shape[0]:
        raise ValueError("Matching matrix dimensions don't match point cloud dimensions")

    if np.count_nonzero(G) > points1.shape[0]:
        raise ValueError("Matching has too many nonzero entries")

    if np.count_nonzero(G) < points1.shape[0]:
        raise ValueError("Matching has too few nonzero entries")

    print("Region accuracy adjusted:", utils.region_accuracy_adjusted(G, faces1, faces2))

    x_ind = 0
    y_ind = 1
    z_ind = 2

    # Ensure numpy arrays
    points1 = np.asarray(points1)
    points2 = np.asarray(points2)
    G = np.asarray(G)

    fig = go.Figure()

    regions1 = utils.faces_to_regions(faces1)
    # regions2 = faces_to_regions(((G / G.max()) @ faces2).astype(int))
    color1 = ["gray" if reg != region_label else "green" for reg in regions1]


    # Plot first set of 3D points
    fig.add_trace(go.Scatter3d(
        x=points1[:, x_ind], y=points1[:, y_ind], z=points1[:, z_ind],
        mode='markers',
        marker=dict(size=size, color=color1),
        name='Points 1'
    ))


    regions2 = utils.faces_to_regions(((G.T / G.max()) @ faces1).astype(int))
    regions2_non_shuffled = utils.faces_to_regions(faces2)
    # regions2 = faces_to_regions(faces2)
    color2 = ["gray" if reg != region_label else "green" for reg in regions2]


    # Plot second set of 3D points
    fig.add_trace(go.Scatter3d(
        x=points2[:, x_ind], y=points2[:, y_ind], z=points2[:, z_ind],
        mode='markers',
        marker=dict(size=size, color= color2),
        name='Points 2'
    ))


    # Draw connections for nonzero G[i, j]
    for i in range(G.shape[0]):
        if regions1[i] != region_label:
            continue
        for j in range(G.shape[1]):
            if G[i, j] != 0:
                p1 = points1[i]
                p2 = points2[j]
                fig.add_trace(go.Scatter3d(
                    x=[p1[x_ind], p2[x_ind]],
                    y=[p1[y_ind], p2[y_ind]],
                    z=[p1[z_ind], p2[z_ind]],
                    mode='lines',
                    line=dict(color="gray", width=2),
                    showlegend=False,
                    opacity=0.3
                ))
                correct_indicator = "green" if regions1[i] == regions2_non_shuffled[j] else "red"
                fig.add_trace(go.Scatter3d(
                    x=[p1[x_ind], p2[x_ind]],
                    y=[p1[y_ind], p2[y_ind]],
                    z=[p1[z_ind], p2[z_ind]],
                    mode='markers',
                    marker=dict(color= correct_indicator, size=size),
                    showlegend=False
                ))

    # Layout styling
    fig.update_layout(
        scene=dict(
            aspectmode="data",
            xaxis=dict(
                visible=False,
                showbackground=False,
                showgrid=False,
                zeroline=False,
                showticklabels=False,
                title=""
            ),
            yaxis=dict(
                visible=False,
                showbackground=False,
                showgrid=False,
                zeroline=False,
                showticklabels=False,
                title=""
            ),
            zaxis=dict(
                visible=False,
                showbackground=False,
                showgrid=False,
                zeroline=False,
                showticklabels=False,
                title=""
            ),
            camera=dict(eye=cam),
        ),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=0, r=0, t=0, b=0),
        width=width,
        height=height,
        showlegend=False,
    )
    return fig

In [14]:
plot_specific_region_connections(
    points1, points2, faces1, faces2, G_aug, "left_shin", cam = dict(x = -2, y = -1.5, z = 1), size = 5,
    height = 1000
)

Region accuracy adjusted: 0.6431535269709544


In [15]:
plot_specific_region_connections(points1, points2, faces1, faces2, G_aug, "left_shin", cam = dict(x = -2, y = -1.5, z = 1))

Region accuracy adjusted: 0.6431535269709544


In [16]:
utils.smplx_vertices_from_amass("datasets/base_smplx_model", "datasets/action_smplx_models/male_run.npz")

(tensor([[[ 3.1893, -3.3055,  1.4503],
          [ 3.1859, -3.3078,  1.4487],
          [ 3.1855, -3.3081,  1.4500],
          ...,
          [ 3.2170, -3.1807,  1.4656],
          [ 3.2161, -3.1829,  1.4664],
          [ 3.2147, -3.1848,  1.4670]],
 
         [[ 3.1750, -3.2878,  1.4544],
          [ 3.1716, -3.2901,  1.4528],
          [ 3.1712, -3.2904,  1.4541],
          ...,
          [ 3.2036, -3.1632,  1.4699],
          [ 3.2027, -3.1654,  1.4706],
          [ 3.2013, -3.1674,  1.4712]],
 
         [[ 3.1605, -3.2689,  1.4610],
          [ 3.1571, -3.2711,  1.4594],
          [ 3.1573, -3.2716,  1.4602],
          ...,
          [ 3.1891, -3.1449,  1.4764],
          [ 3.1881, -3.1471,  1.4771],
          [ 3.1867, -3.1490,  1.4777]],
 
         ...,
 
         [[-3.6990,  3.3768,  1.5700],
          [-3.7024,  3.3743,  1.5686],
          [-3.7029,  3.3743,  1.5698],
          ...,
          [-3.6710,  3.5020,  1.5732],
          [-3.6720,  3.5000,  1.5741],
          [-3.6733

In [17]:
def plot_line_barebones(x, y, color="#000000", width=2):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="lines",
            line=dict(color=color, width=width),
        )
    )

    fig.update_layout(
        xaxis=dict(
            visible=False,
            showgrid=False,
            zeroline=False,
            showticklabels=False,
            title=""
        ),
        yaxis=dict(
            visible=False,
            showgrid=False,
            zeroline=False,
            showticklabels=False,
            title=""
        ),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=0, r=0, t=0, b=0),
        showlegend=False,
        width=600,
        height=400,
    )

    return fig

def plot_lines_barebones(lines, colors=None, width=2):
    """
    lines: list of (x, y) tuples
    colors: list of hex colors or None
    """
    fig = go.Figure()

    for i, (x, y) in enumerate(lines):
        c = colors[i] if colors is not None else None
        fig.add_trace(
            go.Scatter(
                x=x,
                y=y,
                mode="lines",
                line=dict(color=c, width=width),
            )
        )

    fig.update_layout(
        xaxis=dict(visible=False, showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(visible=False, showgrid=False, zeroline=False, showticklabels=False),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=0, r=0, t=0, b=0),
        showlegend=False,
    )

    return fig

def plot_lines_transparent(
    lines,
    colors=None,
    width=2,
    title=None,
    x_title=None,
    y_title=None,
    x_range=None,
    y_range=None,
    show_grid=True,
    show_legend=True,
    linelabels=["Line 1", "Line 2"]
):
    """
    lines: list of (x, y) tuples
    colors: list of colors (hex or rgb strings) or None
    """

    fig = go.Figure()

    for i, (x, y) in enumerate(lines):
        c = colors[i] if colors is not None else None
        fig.add_trace(
            go.Scatter(
                x=x,
                y=y,
                mode="lines",
                line=dict(color=c, width=width),
                name=f"{linelabels[i]}" if show_legend else None,
            )
        )

    fig.update_layout(
        title=title,
        xaxis=dict(
            title=x_title,
            range=x_range,
            showgrid=show_grid,
            zeroline=False,
            showline=True,
            ticks="outside",
        ),
        yaxis=dict(
            title=y_title,
            range=y_range,
            showgrid=show_grid,
            zeroline=False,
            showline=True,
            ticks="outside",
        ),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=60, r=20, t=60 if title else 20, b=50),
        showlegend=show_legend,
        width=700,
        height=400,
    )

    return fig

In [18]:
experiment_aug_accs = np.load("experiment_aug_accs.npy")
experiment_aug_dists = np.load("experiment_aug_dists.npy")
experiment_baseline_accs = np.load("experiment_baseline_accs.npy")
experiment_baseline_dists = np.load("experiment_baseline_dists.npy")

In [ ]:
x = np.arange(20, 91, 10)
plot_lines_transparent(
    [(x, experiment_aug_accs), (x, experiment_baseline_accs)],
    colors = ["green", "red"],
    title = "Region Accuracy: Baseline vs. Ours (Higher is Better)",
    x_title= "Number of Frames in the Future",
    y_title="Region Matching Accuracy",
    show_grid=False,
    linelabels=["Ours", "Baseline"],
    width = 5
)

In [33]:
x = np.arange(20, 91, 10)
plot_lines_transparent(
    [(x, experiment_aug_dists), (x, experiment_baseline_dists)],
    colors = ["green", "red"],
    title = "Region Accuracy: Baseline vs. Ours (Lower is Better)",
    x_title= "Number of Frames in the Future",
    y_title="Region Matching Accuracy",
    show_grid=False,
    linelabels=["Ours", "Baseline"],
    width = 5
)

In [21]:
import numpy as np
import plotly.colors as pc

def values_to_red_blue(values, mid=0.5, colorscale="RdBu", dark_range=(0.05, 0.95)):
    """
    Map float values to a red-blue diverging colorscheme centered at mid (default 0).
    Returns a list of Plotly color strings.

    dark_range: restricts to darker part of the colormap for visibility on light backgrounds.
    """
    values = np.asarray(values, dtype=float)

    # Symmetric range around mid
    max_dev = np.nanmax(np.abs(values - mid))
    if max_dev == 0:
        t = np.full_like(values, 0.5, dtype=float)
    else:
        t = (values - mid) / (2 * max_dev) + 0.5  # map to [0,1]

    lo, hi = dark_range
    t = lo + t * (hi - lo)

    return pc.sample_colorscale(colorscale, t)

In [22]:
colors[0]

'#EF553B'

In [23]:
import plotly.express as px

In [24]:
px.colors.diverging.RdBu

['rgb(103,0,31)',
 'rgb(178,24,43)',
 'rgb(214,96,77)',
 'rgb(244,165,130)',
 'rgb(253,219,199)',
 'rgb(247,247,247)',
 'rgb(209,229,240)',
 'rgb(146,197,222)',
 'rgb(67,147,195)',
 'rgb(33,102,172)',
 'rgb(5,48,97)']

In [25]:
_, idxs = utils.left_right_augmentation(points2, points2.mean(axis = 0) - points1.mean(axis = 0), return_anchor_indices=True)
C = utils.graph_distance_within_cloud_minimally_connected(points2)
colors = values_to_red_blue(C[idxs[0]] - C[idxs[1]], mid = 0, colorscale = px.colors.diverging.Portland)
plot_arr(points2, color = colors, cam = dict(x=0, y=2.5, z=1), size = 5)

In [26]:
np.linalg.norm(points2 - np.array([2.755, -2.557, 0.044]), axis = 1).argmin()

np.int64(359)

In [27]:
colors[644]

'rgb(222, 53, 35)'

In [28]:
P = np.array([
    [0, 1, 0, 0],
    [0.6, 0, 0.4, 0],
    [0, 0.75, 0, 0.25],
    [0, 0, 0.75, 0.25]
])
v = np.array([9/40, 3/8, 1/5, 1/5])
print(v @ P)
v

[0.225 0.375 0.3   0.1  ]


array([0.225, 0.375, 0.2  , 0.2  ])

In [29]:
points1, faces1 = utils.sampled_verts_from_path("datasets/action_smplx_models/female_walk.npz", idx = 0, return_faces=True)
points2, faces2 = utils.sampled_verts_from_path("datasets/action_smplx_models/female_walk.npz", idx = 120, return_faces=True)
a = np.ones(1000) / 1000
b = np.ones(1000) / 1000
M_baseline = ot.dist(points1, points2)
G_baseline = ot.solve(M_baseline, a, b).plan
aug1 = utils.left_right_augmentation(points1, points2.mean(axis = 0) - points1.mean(axis = 0))
aug2 = utils.left_right_augmentation(points2, points2.mean(axis = 0) - points1.mean(axis = 0))
M_aug = ot.dist(aug1, aug2)
G_aug = ot.solve(M_aug, a, b).plan
plot_3d_points_and_connections_region_matched(points1, points2, faces1, faces2, G_aug, cam = dict(x = -2, y = 0.5, z = 1))

Region accuracy adjusted: 0.7678958785249458
